In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchinfo import summary
import torchvision
import matplotlib.pyplot as plt
from pathlib import Path
from torchvision import datasets, transforms, models

In [1]:
!pip install torchinfo

In [3]:
# Bu komutu ilk hücreye hazırlayabilirsin
!cp /content/drive/MyDrive/data/data.zip /content/data.zip

In [4]:
!unzip data.zip

Görüntülenen çıkış son 5000 satıra kısaltıldı.
  inflating: raw/grayscale/Tomato___Leaf_Mold/4b6a59ef-a004-4581-a2c4-d8f94893817e___Crnl_L.Mold 6541.JPG  
  inflating: raw/grayscale/Tomato___Leaf_Mold/9dc10e0a-36ee-4882-84c4-f7c7668550bd___Crnl_L.Mold 7177.JPG  
  inflating: raw/grayscale/Tomato___Leaf_Mold/f3b8f9e3-6c47-4797-9da6-d8b09e19048c___Crnl_L.Mold 9023.JPG  
  inflating: raw/grayscale/Tomato___Leaf_Mold/91eeb305-5a5f-4e72-8258-776ea9ed6338___Crnl_L.Mold 7112.JPG  
  inflating: raw/grayscale/Tomato___Leaf_Mold/3876f39d-ca68-494d-ac22-f71d5d459a66___Crnl_L.Mold 9175.JPG  
  inflating: raw/grayscale/Tomato___Leaf_Mold/f53d879a-6033-4317-8524-e108b10e7656___Crnl_L.Mold 7140.JPG  
  inflating: raw/grayscale/Tomato___Leaf_Mold/51cb012e-4646-447c-b63f-9084a370b781___Crnl_L.Mold 6916.JPG  
  inflating: raw/grayscale/Tomato___Leaf_Mold/27c4fc3e-760a-4d3c-8552-bb5ca4eab3d8___Crnl_L.Mold 6781.JPG  
  inflating: raw/grayscale/Tomato___Leaf_Mold/d8466889-2874-44fd-b6e1-128c328d5b84___Crnl

In [5]:
data_path = Path("splitted_data")

In [6]:
data_path

PosixPath('splitted_data')

In [7]:
import os
def check_data(dir_path):
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"# of directories: {len(dirnames)} and {len(filenames)} images in '{dirpath}'.")

In [8]:
check_data(data_path)

In [9]:
!pip install split-folders

In [10]:
import splitfolders

# ratio=(0.8, 0.1, 0.1) ile veriyi %80, %10, %10 olarak 3 klasöre ayırır.
splitfolders.ratio("raw/color", output="splitted_data", seed=42, ratio=(0.8, 0.1, 0.1), group_prefix=None)

Copying files: 54305 files [00:02, 19566.96 files/s]


In [11]:

# Transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Doğrulama/Test için: Sadece net görüntüler
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# dataset_split klasörünün içindeki train, val ve test yollarını veriyoruz
train_data = datasets.ImageFolder(root='splitted_data/train', transform=train_transform)
val_data   = datasets.ImageFolder(root='splitted_data/val', transform=val_transform)
test_data  = datasets.ImageFolder(root='splitted_data/test', transform=val_transform)

# 3. DataLoader'lar (H100 için batch_size=256 iddialı ve güzel bir rakam)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Başarıyla tanımlandı! Toplam Eğitim Görüntüsü: {len(train_data)}")

Başarıyla tanımlandı! Toplam Eğitim Görüntüsü: 43429


In [12]:
# GPU kontrolü
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan Cihaz: {device}")
class_names = train_data.classes

Kullanılan Cihaz: cuda


In [13]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# 1. train_data (ImageFolder) içindeki tüm etiketleri çekiyoruz
train_labels = train_data.targets

# 'balanced' parametresi veriye göre ağırlık hesaplanmasını sağlar
hesaplanan_agirliklar = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

# 3. Sonucu PyTorch'un ve GPU'nun (CUDA) anlayacağı formata çeviriyoruz
class_weights = torch.tensor(hesaplanan_agirliklar, dtype=torch.float).to(device)

In [14]:
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 333MB/s]


In [15]:
# Tüm katmanlardaki gradyan hesaplamasını kapat (Dondur)
for param in model.parameters():
    param.requires_grad = False

In [16]:
for name, param in model.named_parameters():
    if "classifier" in name:
        param.requires_grad = True

model.classifier = nn.Sequential(
     nn.Dropout(p=0.2, inplace=True),
     nn.Linear(in_features=1280, out_features=len(class_names), bias=True),

)

In [17]:
summary(model, input_size=(256, 3, 224, 224), col_names=["output_size",
"num_params",
"kernel_size",
"trainable"])

Layer (type:depth-idx)                                  Output Shape              Param #                   Kernel Shape              Trainable
EfficientNet                                            [256, 38]                 --                        --                        Partial
├─Sequential: 1-1                                       [256, 1280, 7, 7]         --                        --                        False
│    └─Conv2dNormActivation: 2-1                        [256, 24, 112, 112]       --                        --                        False
│    │    └─Conv2d: 3-1                                 [256, 24, 112, 112]       (648)                     [3, 3]                    False
│    │    └─BatchNorm2d: 3-2                            [256, 24, 112, 112]       (48)                      --                        False
│    │    └─SiLU: 3-3                                   [256, 24, 112, 112]       --                        --                        --
│    └─Sequential

In [28]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)


In [29]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               device: torch.device,
               optimizer: torch.optim.Optimizer):
    # Put model in train mode
    model.train()

    # Setup train loss and train accuracy values
    train_loss, train_acc = 0, 0

    # Loop through data loader data batches
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # 1. Forward pass
        y_pred = model(X)

        # 2. Calculate  and accumulate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # Calculate and accumulate accuracy metrics across all batches
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)

    # Adjust metrics to get average loss and accuracy per batch
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc


def val_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device):
    # Put model in eval mode
    model.eval()

    # Setup test loss and test accuracy values
    val_loss, val_acc = 0, 0

    # Turn on inference context manager
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            # 1. Forward pass
            val_pred_logits = model(X)

            # 2. Calculate and accumulate loss
            loss = loss_fn(val_pred_logits, y)
            val_loss += loss.item()

            # Calculate and accumulate accuracy
            val_pred_labels = val_pred_logits.argmax(dim=1)
            val_acc += ((val_pred_labels == y).sum().item() / len(val_pred_labels))

    # Adjust metrics to get average loss and accuracy per batch
    val_loss = val_loss / len(dataloader)
    val_acc = val_acc / len(dataloader)
    return val_loss, val_acc


def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          val_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          device : torch.device = device,
          epochs: int = 5):
    # 2. Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "val_loss": [],
               "val_acc": []
               }

    # 3. Loop through training and testing steps for a number of epochs
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           device=device,
                                           optimizer=optimizer)
        val_loss, val_acc = val_step(model=model,
                                        dataloader=val_dataloader,
                                        loss_fn=loss_fn,
                                        device=device)

        # 4. Print out what's happening
        print(
            f"Epoch: {epoch + 1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc * 100:.4f} | "
            f"val_loss: {val_loss:.4f} | "
            f"val_acc: {val_acc * 100:.4f}"
        )

        # 5. Update results dictionary
        # Ensure all data is moved to CPU and converted to float for storage
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["val_loss"].append(val_loss.item() if isinstance(val_loss, torch.Tensor) else val_loss)
        results["val_acc"].append(val_acc.item() if isinstance(val_acc, torch.Tensor) else val_acc)

    # 6. Return the filled results at the end of the epochs
    return results

In [30]:
results = train(model=model,
          train_dataloader=train_loader,
          val_dataloader=val_loader,
          optimizer=optimizer,
          loss_fn=loss_fn,
          device=device,
          epochs=3)

Epoch: 1 | train_loss: 0.2561 | train_acc: 92.1720 | val_loss: 0.0654 | val_acc: 97.8125
Epoch: 2 | train_loss: 0.1246 | train_acc: 96.0961 | val_loss: 0.0191 | val_acc: 99.4301
Epoch: 3 | train_loss: 0.0950 | train_acc: 97.0114 | val_loss: 0.0222 | val_acc: 99.4118


In [31]:
torch.save(model.state_dict(), "/content/drive/MyDrive/plant_disease_final_model_99_acc.pth")

In [24]:
torch.save(model.state_dict(), "/content/drive/MyDrive/plant_model_v1.pth")

In [25]:
# Modelin tüm katmanlarını serbest bırak
for param in model.parameters():
    param.requires_grad = True

In [27]:
summary(model, input_size=(64, 3, 224, 224), col_names=["output_size",
"num_params",
"kernel_size",
"trainable"])

Layer (type:depth-idx)                                  Output Shape              Param #                   Kernel Shape              Trainable
EfficientNet                                            [64, 38]                  --                        --                        True
├─Sequential: 1-1                                       [64, 1280, 7, 7]          --                        --                        True
│    └─Conv2dNormActivation: 2-1                        [64, 24, 112, 112]        --                        --                        True
│    │    └─Conv2d: 3-1                                 [64, 24, 112, 112]        648                       [3, 3]                    True
│    │    └─BatchNorm2d: 3-2                            [64, 24, 112, 112]        48                        --                        True
│    │    └─SiLU: 3-3                                   [64, 24, 112, 112]        --                        --                        --
│    └─Sequential: 2-2  

In [33]:
# Modeli değerlendirme moduna alıyoruz
model.eval()

# Senin yazdığın test_step fonksiyonunu kullanarak final skorunu alalım
final_test_loss, final_test_acc = val_step(
    model=model,
    dataloader=test_loader, # En başta ayırdığın %10'luk kısım
    loss_fn=loss_fn,
    device=device
)

print(f"--- FİNAL TEST SONUÇLARI ---")
print(f"Test Kaybı: {final_test_loss:.4f}")
print(f"Test Başarısı (Accuracy): %{final_test_acc * 100:.2f}")

--- FİNAL TEST SONUÇLARI ---
Test Kaybı: 0.0243
Test Başarısı (Accuracy): %99.18


In [34]:
from sklearn.metrics import classification_report

y_preds = []
y_true = []

model.eval()
with torch.inference_mode():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        test_logits = model(X)
        preds = torch.argmax(test_logits, dim=1)
        y_preds.extend(preds.cpu().numpy())
        y_true.extend(y.cpu().numpy())

# Sınıf isimlerini datasetten alalım
class_names = test_data.classes
print(classification_report(y_true, y_preds, target_names=class_names))

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       1.00      1.00      1.00        63
                                 Apple___Black_rot       1.00      1.00      1.00        63
                          Apple___Cedar_apple_rust       1.00      1.00      1.00        28
                                   Apple___healthy       0.99      1.00      1.00       165
                               Blueberry___healthy       1.00      1.00      1.00       151
          Cherry_(including_sour)___Powdery_mildew       1.00      0.97      0.99       106
                 Cherry_(including_sour)___healthy       1.00      0.99      0.99        86
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.74      0.98      0.84        52
                       Corn_(maize)___Common_rust_       1.00      0.98      0.99       120
               Corn_(maize)___Northern_Leaf_Blight       0.98      0.84      0.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def predict_image(image_path):
    # 1. Resmi yükle ve transform uygula
    img = Image.open(image_path)
    img_transformed = transform(img).unsqueeze(0).to(device) # 'transform' senin tanımladığın transformlar olmalı

    # 2. Tahmin yap
    model.eval()
    with torch.inference_mode():
        logits = model(img_transformed)
        probs = torch.softmax(logits, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()

    # 3. Görselleştir
    plt.imshow(img)
    plt.title(f"Tahmin: {class_names[pred_class]} \n Güven: %{probs[0][pred_class]*100:.2f}")
    plt.axis("off")
    plt.show()

# predict_image("test_yaprak.jpg") # Buraya kendi dosya yolunu yaz